In [11]:
import fitz  # PyMuPDF
import os
import re
import time
import sys
import glob

import json


In [34]:
# ================= PENGATURAN =================
SRC_FOLDER = "PDF_Putusan"  # Folder sumber PDF (pastikan folder ini ada sejajar dengan file .ipynb)
DST_FOLDER = "INPUT_TXT"    # Folder tujuan output file TXT
LIMIT = 0                # Batasi jumlah file untuk uji coba (ubah ke 0 untuk proses SEMUA file)
OVERWRITE = True            # True = timpa file jika sudah ada, False = lewati
# ==============================================

def process_pdf(pdf_path, txt_path):
    try:
        doc = fitz.open(pdf_path)
        pages_text = []
        
        for page in doc:
            raw_text = page.get_text("text")
            processed_lines = []
            
            # Normalisasi spasi antar kata per baris
            for line in raw_text.splitlines():
                cleaned_line = re.sub(r"[ \t]+", " ", line).strip()
                if cleaned_line:  # Buang baris kosong
                    processed_lines.append(cleaned_line)
            
            if processed_lines:
                pages_text.append("\n".join(processed_lines))
                
        final_text = "\n\n".join(pages_text)
        
        if not final_text.strip():
            print(f"ERROR (Teks kosong): {pdf_path}", file=sys.stderr)
            return False, 0
            
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(final_text)
            
        return True, len(final_text)
        
    except Exception as e:
        print(f"ERROR (Gagal baca): {pdf_path} - {e}", file=sys.stderr)
        return False, 0

In [35]:
# Eksekusi Utama
os.makedirs(DST_FOLDER, exist_ok=True)
pdf_files = [f for f in os.listdir(SRC_FOLDER) if f.lower().endswith('.pdf')]

if LIMIT > 0:
    pdf_files = pdf_files[:LIMIT]
    
sukses = gagal = total_karakter = 0
start_time = time.time()

print(f"Memulai pemrosesan {len(pdf_files)} file PDF...")

for filename in pdf_files:
    pdf_path = os.path.join(SRC_FOLDER, filename)
    txt_name = f"T-{os.path.splitext(filename)[0]}.txt"
    txt_path = os.path.join(DST_FOLDER, txt_name)
    
    if os.path.exists(txt_path) and not OVERWRITE:
        continue
        
    success, char_count = process_pdf(pdf_path, txt_path)
    if success:
        sukses += 1
        total_karakter += char_count
    else:
        gagal += 1
        
print(f"\n[REKAP] Sukses: {sukses} | Gagal: {gagal} | Karakter: {total_karakter} | Waktu: {time.time()-start_time:.2f}s")

Memulai pemrosesan 273 file PDF...

[REKAP] Sukses: 273 | Gagal: 0 | Karakter: 19409242 | Waktu: 27.79s


In [5]:
# Menampilkan isi file TXT pertama yang berhasil dibuat
txt_files = glob.glob(f"{DST_FOLDER}/*.txt")

if txt_files:
    print(f"Membaca file: {txt_files[0]}\n" + "-"*40)
    with open(txt_files[0], "r", encoding="utf-8") as f:
        # Menampilkan 30 baris pertama
        for _ in range(30):
            print(f.readline().rstrip('\n'))
else:
    print("Belum ada file TXT yang dihasilkan.")

Membaca file: INPUT_TXT/T-3162_Pid.B_2017_PN SBY.txt
----------------------------------------
hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
P U T U S A N
No. 3150/Pid.B/2017/PN. SBY.
“DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA”
Pengadilan Negeri Surabaya yang mengadili perkara-perkara pidana dengan
acara biasa pada tingkat pertama telah menjatuhkan Putusan sebagai berikut dalam
perkara Terdakwa :
Nama Lengkap
: DONI KUSUMA Bin MATMUDJI ;
Tempat lahir
: Surabaya ;
Umur/tanggal lahir
: 32 Tahun / 26 April 1985 ;
Jenis Kelamin
: Laki-laki ;
Kebangsaan
: Indonesia ;
Tempat tinggal
: Tambak Gringsing I/3, Surabaya ;
Agama
: Islam ;
P e k e r j a a n
: Swasta (Kuli Bangunan) ;
Pendidikan


## EntityGenerator

In [30]:
import re
import json

def ekstrak_teks_putusan(path_txt):
    data = {
        "nomor_putusan": None,
        "nama_terdakwa_1": None,
        "nama_terdakwa_2": None,
        "nama_terdakwa_3": None,
        "tuntutan_pidana": None,
        "tuntutan_kuhp": None,
        "tuntutan_hukuman": None,
        "putusan_pidana": None,
        "putusan_hukuman": None,
        "tanggal_putusan": None,
        "hakim_ketua": None,
        "hakim_anggota_1": None,
        "hakim_anggota_2": None,
        "panitera": None,
        "penuntut_umum": None
    }
    
    try:
        with open(path_txt, 'r', encoding='utf-8', errors='ignore') as f:
            raw_text = f.read()
            
        # 1. PEMBERSIHAN NOISE (Membersihkan teks dari Footer PDF Mahkamah Agung)
        teks = re.sub(r'\s+', ' ', raw_text).lower()
        
        # Hapus blok disclaimer panjang yang sering memutus kalimat
        teks = re.sub(r'halaman \d+ dari \d+ putusan.*?ext\.318\)', ' ', teks)
        teks = re.sub(r'halaman \d+ hkama.*?putusan\.mahkamahagung\.go\.id', ' ', teks)
        teks = re.sub(r'direktori putusan mahkamah agung republik indonesia', ' ', teks)
        teks = re.sub(r'\s+', ' ', teks) # Rapikan spasi sisa
        
        # 2. Nomor Putusan (Diperlebar agar menangkap spasi pada format "PN. SBY.")
        m_no = re.search(r'no\.?\s*(\d+/[a-z\./]+/\d{4}/[a-z\.\s]+?)\s*(?:“|"|demi)', teks)
        if m_no: data["nomor_putusan"] = m_no.group(1).strip().upper()
            
        # 3. Nama Terdakwa
        list_nama = re.findall(r'nama\s*(?:lengkap|terdakwa)?\s*:\s*(.*?)\s*(?:;|tempat\s*lahir)', teks)
        list_terdakwa = []
        for n in list_nama:
            n_clean = n.strip().upper()
            if n_clean and len(n_clean) > 2 and n_clean not in list_terdakwa:
                list_terdakwa.append(n_clean)
                
        if len(list_terdakwa) > 0: data["nama_terdakwa_1"] = list_terdakwa[0]
        if len(list_terdakwa) > 1: data["nama_terdakwa_2"] = list_terdakwa[1]
        if len(list_terdakwa) > 2: data["nama_terdakwa_3"] = list_terdakwa[2]

        # 4. Tuntutan
        m_tuntutan = re.search(r'menuntut\s+agar\s+supaya.*?memutuskan\s*sebagai\s+berikut\s*:(.*?)menimbang', teks)
        if m_tuntutan:
            t_teks = m_tuntutan.group(1)
            tp = re.search(r'tindak\s+pidana\s+(.*?)\s+sebagaimana', t_teks)
            if tp: data["tuntutan_pidana"] = tp.group(1).title()
            
            kuhp = re.search(r'pasal\s+(.*?kuhp[a-z]*)', t_teks)
            if kuhp: data["tuntutan_kuhp"] = "Pasal " + kuhp.group(1).title()
            
            hk = re.search(r'selama\s+(.*?)\s+(?:dengan\s+)?dikurangi', t_teks)
            if hk: data["tuntutan_hukuman"] = hk.group(1).title()

        # 5. Putusan
        # Mencari blok setelah kata mengadili, jika tidak ada cari pola terbukti bersalah secara umum
        m_putusan = re.search(r'm\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i(.*?)demikian', teks)
        if not m_putusan:
            m_putusan = re.search(r'mengadili\s*[:,]*(.*?)demikian', teks)
            
        p_teks = m_putusan.group(1) if m_putusan else teks # fallback ke seluruh teks jika gagal
        
        # Ekstrak dari p_teks
        tp_putusan = re.search(r'terbukti\s+secara\s+sah.*?(?:melakukan\s+)?tindak\s+pidana\s+"?(.*?)"?\s*(?:sebagaimana|;|\.)', p_teks)
        if tp_putusan: data["putusan_pidana"] = tp_putusan.group(1).strip().title()
        
        hk_putusan = re.search(r'menjatuhkan\s+pidana.*?selama\s+(.*?)\s+(?:dan|dengan|dikurangi|;|\.)', p_teks)
        if hk_putusan: data["putusan_hukuman"] = hk_putusan.group(1).strip().title()

        # 6. Susunan Majelis Hakim (Dibuat Independen agar tidak bergantung pada 1 struktur kalimat penuh)
        m_tgl = re.search(r'diputuskan.*?(?:pada\s+hari.*?)?tanggal\s*(.*?)\s*,?\s*oleh', teks)
        if m_tgl: data["tanggal_putusan"] = m_tgl.group(1).title()
        
        m_ketua = re.search(r'oleh\s*(?:kami)?\s*[:,]?\s*([a-z\.\,\s]+?)\s*,?\s*(?:selaku|sebagai)\s*hakim ketua', teks)
        if m_ketua: data["hakim_ketua"] = m_ketua.group(1).strip().title()
        
        # Mengambil anggota dari "Hakim Ketua... , X dan Y masing-masing sebagai Hakim Anggota"
        m_anggota = re.search(r'hakim ketua.*?,?\s*([a-z\.\,\s]+?)\s*dan\s*([a-z\.\,\s]+?)\s*,?\s*masing-masing\s*(?:selaku|sebagai)\s*hakim anggota', teks)
        if m_anggota:
            data["hakim_anggota_1"] = m_anggota.group(1).strip().title()
            data["hakim_anggota_2"] = m_anggota.group(2).strip().title()
            
        m_panitera = re.search(r'dibantu\s*(?:oleh)?\s*[:,]?\s*([a-z\.\,\s]+?)\s*,?\s*(?:sebagai)?\s*panitera', teks)
        if m_panitera: data["panitera"] = m_panitera.group(1).strip().title()
        
        m_jaksa = re.search(r'dihadiri\s*(?:oleh)?\s*[:,]?\s*([a-z\.\,\s]+?)\s*,?\s*(?:sebagai)?\s*(?:penuntut umum|jaksa)', teks)
        if m_jaksa: data["penuntut_umum"] = m_jaksa.group(1).strip().title()

        return data
    except Exception as e:
        return {"_error": str(e)}

In [31]:
# ====== UJI COBA ======
if __name__ == "__main__":
    # Ganti string di bawah dengan nama file PDF/TXT Anda
    hasil = ekstrak_teks_putusan("INPUT_TXT/T-3162_Pid.B_2017_PN SBY.txt")
    print(json.dumps(hasil, indent=2))

{
  "nomor_putusan": "3150/PID.B/2017/PN. SBY.",
  "nama_terdakwa_1": "DONI KUSUMA BIN MATMUDJI",
  "nama_terdakwa_2": null,
  "nama_terdakwa_3": null,
  "tuntutan_pidana": "Pencurian Dengan Pemberatan",
  "tuntutan_kuhp": "Pasal 363 Ayat 2 Kuhpidana",
  "tuntutan_hukuman": "4 (Empat) Tahun Penjara",
  "putusan_pidana": null,
  "putusan_hukuman": "4 (Empat) Tahun Penjara",
  "tanggal_putusan": null,
  "hakim_ketua": "F.X Hanung Dwi W., Sh.,Mh.",
  "hakim_anggota_1": "Majelis Isjuaedi, Sh.Mh.",
  "hakim_anggota_2": "Agus Hamzah, Sh.Mh.",
  "panitera": "Aris Andriana, Sh.Mh.",
  "penuntut_umum": "Oky Muji Astuti, Sh."
}


In [21]:
# ==========================================
# CELL 2: PERINTAH UJI BERJALAN
# ==========================================

# 1. Uji 1 file (Pastikan ada file contoh-in.txt di folder yang sama)
print("=== UJI 1 FILE ===")
hasil_1 = ekstrak("INPUT_TXT/T-3162_Pid.B_2017_PN SBY.txt")
print(json.dumps(hasil_1, indent=2))

# 2. Uji semua file di 1 folder (Contoh folder: INPUT_TXT)
print("\n=== UJI BATCH (FOLDER) ===")
folder_input = "INPUT_TXT"
if os.path.exists(folder_input):
    for filename in os.listdir(folder_input)[:3]: # limit 3 file
        if filename.endswith(".txt"):
            path = os.path.join(folder_input, filename)
            print(f"Ekstrak {filename}:", "Sukses" if "_error" not in ekstrak(path) else "Error")

# 3. Uji toleransi file rusak
print("\n=== UJI TOLERANSI FILE RUSAK ===")
with open("file_rusak.txt", "wb") as f: # Buat file non-utf8
    f.write(b'\xff\xfe\x00\x00')
hasil_rusak = ekstrak("file_rusak.txt")
print(json.dumps(hasil_rusak, indent=2))

=== UJI 1 FILE ===
{
  "nomor_putusan": "3150/pid.b/2017/pn. sby.",
  "nama_terdakwa_1": null,
  "nama_terdakwa_2": null,
  "nama_terdakwa_3": null,
  "tuntutan_pidana": null,
  "tuntutan_kuhp": null,
  "tuntutan_hukuman": null,
  "putusan_pidana": null,
  "putusan_hukuman": null,
  "tanggal_putusan": null,
  "hakim_ketua": null,
  "hakim_anggota_1": null,
  "hakim_anggota_2": null,
  "panitera": null,
  "penuntut_umum": null
}

=== UJI BATCH (FOLDER) ===
Ekstrak T-3162_Pid.B_2017_PN SBY.txt: Sukses

=== UJI TOLERANSI FILE RUSAK ===
{
  "_error": "'utf-8' codec can't decode byte 0xff in position 0: invalid start byte"
}
